In [0]:
# Análise Exploratória de Dados (EDA)
# Squad 1 - Dupla 5
# Tabelas: ecommerce_clientes

In [0]:
%pip install azure-storage-file-datalake azure-identity pandas pyarrow
dbutils.library.restartPython()

In [0]:
import pandas as pd

In [0]:
tabela_produtos = "ecommerce_clientes"


print("Tabela:", tabela_clientes)


In [0]:
df_clientes = None

print("DataFrames preparados para leitura dos dados.")

In [0]:
storage_account_name = "internshipdatalake"
container_name = "raw"

adls_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
)

caminho_real_time = f"{adls_path}real-time-data/"

print("Diretório de referência:")
print(caminho_real_time)

In [0]:
from dotenv import load_dotenv
import os
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
import pandas as pd
from io import BytesIO

load_dotenv(
    '.env',
    override=True,
)

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

credential = ClientSecretCredential(tenant_id, client_id, client_secret)
service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential,
)
fs_client = service_client.get_file_system_client(container_name)

# Localizar o arquivo ecommerce_clientes.parquet
arquivos = fs_client.get_paths(path="real-time-data")

arquivo_clientes = None

for arquivo in arquivos:
    if arquivo.name.endswith("ecommerce_clientes.parquet"):
        arquivo_clientes = arquivo.name
        break

print("Arquivo encontrado:", arquivo_clientes)




In [0]:
lista_df = []

for caminho in arquivos_clientes:
    file_client = fs_client.get_file_client(caminho)

    download = file_client.download_file()
    dados = download.readall()

    df_temp = pd.read_parquet(BytesIO(dados))

    print(caminho, "->", len(df_temp), "linhas")

    lista_df.append(df_temp)

# Junta os 6 arquivos
df_clientes_completo = pd.concat(lista_df, ignore_index=True)

print("\nTotal de linhas:", len(df_clientes_completo))

display(df_clientes_completo)

In [0]:
from pyspark.sql import functions as F

def analisar_qualidade_dataframe(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    print(f"=== Qualidade dos dados: {nome} ===")

    print("\nValores nulos por coluna:")
    df.select([
        F.sum(F.col(col).isNull().cast("int")).alias(col)
        for col in df.columns
    ]).show()

    total_registros = df.count()
    registros_unicos = df.dropDuplicates().count()
    duplicados = total_registros - registros_unicos

    print("\nDuplicidades:")
    print(f"Total de registros: {total_registros}")
    print(f"Registros duplicados: {duplicados}")

In [0]:
print("Quantidade total de linhas:", len(df_clientes_completo))

print("\nColunas:")
print(df_clientes_completo.columns.tolist())

print("\nValores nulos por coluna:")
print(df_clientes_completo.isnull().sum())

print("\nQuantidade de linhas duplicadas:")
print(df_clientes_completo.duplicated().sum())


In [0]:
def analisar_schema_dataframe(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    print(f"=== Schema: {nome} ===")
    df.printSchema()

In [0]:
def analisar_estatisticas_dataframe(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    print(f"=== Estatísticas descritivas: {nome} ===")
    df.describe().show()

In [0]:
def analisar_valores_distintos(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    print(f"=== Valores distintos: {nome} ===")

    for coluna in df.columns:
        quantidade = df[coluna].nunique()
        print(f"{coluna}: {quantidade} valores distintos")

In [0]:
analisar_valores_distintos(
    df_clientes_completo,
    "ecommerce_clientes"
)

In [0]:
def analisar_duplicidades_coluna(df, coluna, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    duplicados = df[df.duplicated(subset=[coluna], keep=False)]

    print(f"=== Duplicidades: {nome} / {coluna} ===")
    print(f"Total de registros: {len(df)}")
    print(f"Registros com {coluna} duplicado: {len(duplicados)}")

    if len(duplicados) > 0:
        display(duplicados.sort_values(by=coluna))
    else:
        print("Nenhuma duplicidade encontrada.")

In [0]:
analisar_duplicidades_coluna(
    df_clientes_completo,
    "id_cliente",
    "ecommerce_clientes"
)

Salvando pipeline de carga (Data Load) no Azure SQL Server.

In [0]:
# Converter o DataFrame Pandas para Spark
df_clientes_spark = spark.createDataFrame(df_clientes_completo)

print("DataFrame convertido para Spark.")
print("Quantidade de registros:", df_clientes_spark.count())

df_clientes_spark.printSchema()

In [0]:
df_clientes_spark.write \
    .format("sqlserver") \
    .option("host", jdbc_hostname) \
    .option("port", "1433") \
    .option("database", jdbc_database) \
    .option("dbtable", "squad1.ecommerce_clientes") \
    .option("user", jdbc_username) \
    .option("password", jdbc_password) \
    .option("encrypt", "true") \
    .mode("overwrite") \
    .save()

print("Tabela squad1.ecommerce_clientes gravada com sucesso.")

In [0]:
df_clientes_spark.write

In [0]:
jdbc_hostname = os.getenv("SQL_HOST")
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

df_clientes_spark.write \
    .format("sqlserver") \
    .option("host", jdbc_hostname) \
    .option("port", "1433") \
    .option("database", jdbc_database) \
    .option("dbtable", "squad1.ecommerce_clientes") \
    .option("user", jdbc_username) \
    .option("password", jdbc_password) \
    .option("encrypt", "true") \
    .mode("overwrite") \
    .save()

print("✅ Tabela squad1.ecommerce_clientes gravada com sucesso!")

